# MATH 616: Computational Linear Algebra
## Coding Lab — Direct Methods for Solving Linear Systems

This notebook is the coding companion to the *Direct Methods* lecture. We implement every algorithm from the slides in Python, check each one against the worked examples, and then **measure** how the cost of a solve grows with the size $n$ of the system.

### Learning goals
By the end of the lab you should be able to:

- implement forward and backward substitution and explain why they cost $O(n^2)$;
- implement naive Gaussian elimination and reproduce both of its failure modes (a zero pivot and a tiny pivot);
- implement partial pivoting and scaled-column partial pivoting, and explain when the second one matters;
- count the floating-point operations of an elimination and match the count to the formula $\approx n^3/3$;
- solve many right-hand sides at once, and compute $A^{-1}$ from $[A \mid I]$;
- **time** the algorithms for large $n$ and read the exponent off a log–log plot.

### How to use this notebook
Run the cells from top to bottom. Cells marked **✏️ Your turn** are where you change a value (usually `n`) and re-run.

## 0. Imports

In [ ]:
import time
import math
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)
rng = np.random.default_rng(616)

# Part I. Why we need direct methods

## 1. Cramer's rule is exact — and unusable

Cramer's rule gives $x_i = \det(A_i)/\det(A)$, where $A_i$ is $A$ with column $i$ replaced by $b$. Computing a determinant by **cofactor expansion** costs $O(n!)$ operations. Let's implement it, count its multiplications, and see the factorial wall for ourselves.

In [ ]:
def det_cofactor(A, counter):
    """Determinant by cofactor expansion along the first row.
    counter is a one-element list; counter[0] accumulates the multiplications.
    """
    n = A.shape[0]
    if n == 1:
        return A[0, 0]
    total = 0.0
    for j in range(n):
        minor = np.delete(A[1:], j, axis=1)     # drop row 0 and column j
        counter[0] += 1
        total += (-1) ** j * A[0, j] * det_cofactor(minor, counter)
    return total


def cramer(A, b):
    """Solve A x = b by Cramer's rule. Returns x and the multiplication count."""
    n = A.shape[0]
    counter = [0]
    detA = det_cofactor(A, counter)
    x = np.zeros(n)
    for i in range(n):
        Ai = A.copy()
        Ai[:, i] = b
        x[i] = det_cofactor(Ai, counter) / detA
    return x, counter[0]

In [ ]:
A = np.array([[ 2.,  3.,  -5.,  0.],
              [ 2., -1.,   6.,  1.],
              [ 1.,  5., -16., -8.],
              [ 1.,  1.,   0.,  1.]])
b = np.array([1., -1., 13., 13.])

x, mults = cramer(A, b)
print("Cramer solution x =", x)
print("check  A x - b    =", A @ x - b)
print("multiplications   =", mults)

Now watch the count and the wall-clock time explode as $n$ grows. (We stop at $n = 8$. Try $n = 9$ (about 10 seconds), then predict $n = 10$ before you run it.)

In [ ]:
print(f"{'n':>3} {'multiplications':>16} {'time (s)':>10}")
for n in range(2, 9):     # <-- try range(2, 10)
    A = rng.standard_normal((n, n))
    b = rng.standard_normal(n)
    t0 = time.perf_counter()
    _, mults = cramer(A, b)
    t = time.perf_counter() - t0
    print(f"{n:>3} {mults:>16,} {t:>10.4f}")

Each increase of $n$ by one multiplies the time by roughly $n$. Extrapolate: how long would $n = 20$ take on your laptop? (Hint: $20!/8! \approx 6 \times 10^{13}$.)

Gaussian elimination solves the same system in about $n^3/3$ operations. The rest of this notebook builds that algorithm.

# Part II. Triangular systems

## 2. Backward substitution

For an upper-triangular $U$, solve the last equation first and work upward:

$$x_i = \frac{1}{u_{ii}}\Big(b_i - \sum_{j>i} u_{ij}x_j\Big), \qquad i = n, n-1, \dots, 1.$$

As on the slides, `b` is a 2-D array of shape `(n, m)`, so one call solves $m$ right-hand sides at once.

In [ ]:
def backward(U, b):
    """Solve U x = b for an upper-triangular U.
    U is (n, n), b is (n, m), and the result x is (n, m).
    """
    n, m = b.shape
    x = np.zeros((n, m))
    x[n-1] = b[n-1] / U[n-1, n-1]
    for i in range(n-2, -1, -1):
        x[i] = (b[i] - U[i, i+1:] @ x[i+1:]) / U[i, i]
    return x

**Example (slide: upper-triangular systems).** The expected solution is $x = (1, 1, 0, 2)$.

In [ ]:
U = np.array([[3.,  1., 1., 0.],
              [0., -2., 0., 1.],
              [0.,  0., 3., 2.],
              [0.,  0., 0., 4.]])
b = np.array([[4.], [0.], [4.], [8.]])     # column vector, shape (4, 1)

x = backward(U, b)
print("x =", x.ravel())

## 3. Forward substitution

For a lower-triangular $L$, start at the top:

$$x_i = \frac{1}{\ell_{ii}}\Big(b_i - \sum_{j<i} \ell_{ij}x_j\Big), \qquad i = 1, 2, \dots, n.$$

In [ ]:
def forward(L, b):
    """Solve L x = b for a lower-triangular L.
    L is (n, n), b is (n, m), and the result x is (n, m).
    """
    n, m = b.shape
    x = np.zeros((n, m))
    x[0] = b[0] / L[0, 0]
    for i in range(1, n):
        x[i] = (b[i] - L[i, :i] @ x[:i]) / L[i, i]
    return x

**Example (slide: lower-triangular systems).** The expected solution is $x = (-1, -2, 0, 0)$.

In [ ]:
L = np.array([[ 5., 0., 0.,  0.],
              [ 2., 1., 0.,  0.],
              [-1., 0., 1.,  0.],
              [-2., 3., 0., -4.]])
b = np.array([[-5.], [-4.], [1.], [-4.]])

x = forward(L, b)
print("x =", x.ravel())

**Several right-hand sides at once.** Because `b` is 2-D, stacking columns costs nothing extra in code:

In [ ]:
B = np.array([[-5.,  5.],
              [-4.,  3.],
              [ 1., -1.],
              [-4.,  0.]])
X = forward(L, B)
print("X =\n", X)
print("residual L X - B =\n", L @ X - B)

# Part III. Naive Gaussian elimination

## 4. The algorithm

At step $i$, use row $i$ to clear every entry below the pivot $a_{ii}$:

$$E_j \leftarrow E_j - \ell_j E_i, \qquad \ell_j = \frac{a_{ji}}{a_{ii}}, \qquad j = i+1, \dots, n.$$

After $n-1$ steps the system is upper triangular, and one backward substitution finishes the solve.

In [ ]:
def gausselim(A, b):
    """Solve A x = b by Gaussian elimination, no pivoting."""
    A = A.astype(float).copy()   # do not overwrite the caller's
    b = b.astype(float).copy()   # arrays
    n = A.shape[0]
    for i in range(n):
        for j in range(i+1, n):
            c = A[j, i] / A[i, i]      # needs A[i, i] != 0
            A[j, i:] -= c * A[i, i:]
            b[j]     -= c * b[i]
    return backward(A, b)

It is useful to also have a version that **returns the triangular system**, so we can compare with the hand computation on the slides.

In [ ]:
def eliminate(A, b):
    """Naive elimination only. Returns the upper-triangular U and the updated b."""
    A = A.astype(float).copy()
    b = b.astype(float).copy()
    n = A.shape[0]
    for i in range(n):
        for j in range(i+1, n):
            c = A[j, i] / A[i, i]
            A[j, i:] -= c * A[i, i:]
            b[j]     -= c * b[i]
    return A, b

**Example (slides: the 4 × 4 elimination).** The slides reach the triangular system below and the solution $x = (-1, 2, 0, 1)$.

In [ ]:
A = np.array([[ 1.,  1.,  0.,  3.],
              [ 2.,  1., -1.,  1.],
              [ 3., -1., -1.,  2.],
              [-1.,  2.,  3., -1.]])
b = np.array([[4.], [1.], [-3.], [4.]])

U, bt = eliminate(A, b)
print("[U | b~] =\n", np.hstack([U, bt]))

x = gausselim(A, b)
print("\nx =", x.ravel())
print("residual ||A x - b|| =", np.linalg.norm(A @ x - b))

## 5. Breakdown 1: a zero pivot

The matrix below is invertible, but after the first step the pivot $a_{22}$ is exactly zero. NumPy does not raise an exception on floating-point division by zero — it returns `inf` or `nan` and prints a warning.

In [ ]:
A = np.array([[1., -3.,  0.],
              [1., -3.,  5.],
              [2.,  1., -1.]])
b = np.array([[-1.], [4.], [4.]])

print("det(A) =", np.linalg.det(A))
x = gausselim(A, b)
print("naive Gaussian elimination gives x =", x.ravel())

Swapping rows 2 and 3 repairs it — the correct answer is $x = (2, 1, 1)$:

In [ ]:
x = gausselim(A[[0, 2, 1]], b[[0, 2, 1]])
print("after swapping rows 2 and 3, x =", x.ravel())

## 6. Breakdown 2: a tiny pivot

The slides carried four significant digits with $\varepsilon = 10^{-8}$. NumPy uses double precision (about 16 digits), so to see the same effect we need a smaller $\varepsilon$. The exact solution is $x_1 = x_2 = 1$ for every $\varepsilon$.

$$\begin{bmatrix}\varepsilon & 1\\ 2 & 3\end{bmatrix}\begin{bmatrix}x_1\\x_2\end{bmatrix}=\begin{bmatrix}1+\varepsilon\\5\end{bmatrix}$$

In [ ]:
def tiny_pivot_system(eps):
    A = np.array([[eps, 1.],
                  [2.,  3.]])
    b = np.array([[1. + eps], [5.]])
    return A, b

print(f"{'eps':>8} {'x1':>22} {'x2':>22}")
for k in range(2, 20, 2):
    eps = 10.0 ** (-k)
    A, b = tiny_pivot_system(eps)
    x = gausselim(A, b).ravel()
    print(f"{eps:>8.0e} {x[0]:>22.15f} {x[1]:>22.15f}")

**Observe:** $x_2$ stays correct, while $x_1$ loses more and more digits and eventually becomes $0$ — with no warning. The matrix is perfectly well-conditioned; the algorithm is to blame.

In [ ]:
A, b = tiny_pivot_system(1e-20)
print("condition number of A:", np.linalg.cond(A))
print("naive GE     :", gausselim(A, b).ravel())
print("swap rows    :", gausselim(A[[1, 0]], b[[1, 0]]).ravel())

# Part IV. Pivoting strategies

## 7. Partial pivoting

At step $i$, choose the row $p \ge i$ with the largest $|a_{pi}|$, interchange rows $p$ and $i$, then eliminate. Every multiplier then satisfies $|\ell_j| \le 1$.

Following the slides, we do not move rows in memory; we record the interchanges in a **permutation vector** `IP`. (Python counts from 0, so the slides' `IP = [3, 1, 2]` appears here as `[2, 0, 1]`.) For clarity the implementation below does swap rows, and keeps `IP` alongside so we can see the interchanges.

In [ ]:
def gauss_pp(A, b, return_perm=False):
    """Solve A x = b by Gaussian elimination with partial pivoting."""
    A = A.astype(float).copy()
    b = b.astype(float).copy()
    n = A.shape[0]
    IP = np.arange(n)
    for i in range(n-1):
        p = i + np.argmax(np.abs(A[i:, i]))   # argmax returns the first (smallest) index
        if p != i:
            A[[i, p]] = A[[p, i]]
            b[[i, p]] = b[[p, i]]
            IP[[i, p]] = IP[[p, i]]
        for j in range(i+1, n):
            c = A[j, i] / A[i, i]             # |c| <= 1
            A[j, i:] -= c * A[i, i:]
            b[j]     -= c * b[i]
    x = backward(A, b)
    return (x, IP) if return_perm else x

**Example (slides: partial pivoting).** Expected: $x = (1, 1, -1)$ and `IP = [3, 1, 2]` in the slides' 1-based numbering.

In [ ]:
A = np.array([[2.,  1.,  0.],
              [1., -1.,  4.],
              [3., -1., -2.]])
b = np.array([[3.], [-4.], [4.]])

x, IP = gauss_pp(A, b, return_perm=True)
print("x  =", x.ravel())
print("IP =", IP, "  (1-based:", IP + 1, ")")

Partial pivoting also rescues both earlier failures:

In [ ]:
A0 = np.array([[1., -3., 0.], [1., -3., 5.], [2., 1., -1.]])
b0 = np.array([[-1.], [4.], [4.]])
print("zero pivot  :", gauss_pp(A0, b0).ravel())

A1, b1 = tiny_pivot_system(1e-20)
print("tiny pivot  :", gauss_pp(A1, b1).ravel())

## 8. Partial pivoting can be fooled by scaling

Multiply the first equation of the tiny-pivot system by $10^4/\varepsilon$. The solution is still $x = (1, 1)$, but now $|a_{11}| = 10^4 > 2$, so partial pivoting **keeps** row 1 as the pivot row.

In [ ]:
def rescaled_system(eps):
    A, b = tiny_pivot_system(eps)
    s = 1e4 / eps
    A[0] *= s
    b[0] *= s
    return A, b

A, b = rescaled_system(1e-20)
print("A =\n", A)
print("partial pivoting:", gauss_pp(A, b).ravel())

## 9. Scaled-column partial pivoting

Compute the row scales $s_i = \max_j |a_{ij}|$ **once**, from the original matrix. At step $i$ pick the row $p$ that maximizes $|a_{pi}|/s_p$. The scales travel with their rows when rows are interchanged.

In [ ]:
def gauss_spp(A, b, return_perm=False):
    """Solve A x = b by Gaussian elimination with scaled-column partial pivoting."""
    A = A.astype(float).copy()
    b = b.astype(float).copy()
    n = A.shape[0]
    s = np.max(np.abs(A), axis=1)             # row scales, computed once
    IP = np.arange(n)
    for i in range(n-1):
        p = i + np.argmax(np.abs(A[i:, i]) / s[i:])
        if p != i:
            A[[i, p]] = A[[p, i]]
            b[[i, p]] = b[[p, i]]
            s[[i, p]] = s[[p, i]]
            IP[[i, p]] = IP[[p, i]]
        for j in range(i+1, n):
            c = A[j, i] / A[i, i]
            A[j, i:] -= c * A[i, i:]
            b[j]     -= c * b[i]
    x = backward(A, b)
    return (x, IP) if return_perm else x

**Example (slides: scaled-column pivoting).** Here $s = (2, 4, 3)$; there is no interchange at step 1 (unlike partial pivoting), and rows 2 and 3 are interchanged at step 2. Expected: $x = (1, 1, -1)$.

In [ ]:
A = np.array([[2.,  1.,  0.],
              [1., -1.,  4.],
              [3., -1., -2.]])
b = np.array([[3.], [-4.], [4.]])

x, IP = gauss_spp(A, b, return_perm=True)
print("x  =", x.ravel())
print("IP =", IP, "  (1-based:", IP + 1, ")")

And on the rescaled system that fooled partial pivoting:

In [ ]:
A, b = rescaled_system(1e-20)
print("partial pivoting       :", gauss_pp(A, b).ravel())
print("scaled-column pivoting :", gauss_spp(A, b).ravel())

## 10. Comparing the three methods on random systems

On a "nice" random matrix all three methods usually agree. The differences show up on matrices with small pivots. Below we compare the relative error $\|x - x_{\text{true}}\| / \|x_{\text{true}}\|$ on a random matrix whose first row has been made tiny.

In [ ]:
def rel_err(x, x_true):
    return np.linalg.norm(x - x_true) / np.linalg.norm(x_true)

n = 50
x_true = np.ones((n, 1))

A_nice = rng.standard_normal((n, n))
A_bad = A_nice.copy()
A_bad[0, 0] = 1e-14                    # a tiny first pivot

print(f"{'matrix':>8} {'naive':>12} {'partial':>12} {'scaled':>12}")
for name, M in [("nice", A_nice), ("bad", A_bad)]:
    rhs = M @ x_true
    errs = [rel_err(f(M, rhs), x_true) for f in (gausselim, gauss_pp, gauss_spp)]
    print(f"{name:>8} " + " ".join(f"{e:>12.2e}" for e in errs))

# Part V. Counting the work

## 11. An instrumented elimination

The slides derive exact counts for elimination on the augmented matrix $[A \mid b]$ followed by backward substitution:

| | multiply / divide | add / subtract |
|---|---|---|
| elimination | $(2n^3 + 3n^2 - 5n)/6$ | $(n^3 - n)/3$ |
| backward substitution | $(n^2 + n)/2$ | $(n^2 - n)/2$ |
| **whole solve** | $n^3/3 + n^2 - n/3$ | $n^3/3 + n^2/2 - 5n/6$ |

Let's count the operations as the algorithm actually performs them and check the formulas. (We skip the entry $a_{ji}$ itself, which is set to zero rather than computed.)

In [ ]:
def gausselim_count(A, b):
    """Naive Gaussian elimination + backward substitution for one right-hand side,
    counting the floating-point operations. Returns x, (mult/div count), (add/sub count).
    """
    A = A.astype(float).copy()
    b = b.astype(float).ravel().copy()
    n = A.shape[0]
    md_ops = 0
    as_ops = 0
    # elimination
    for i in range(n-1):
        for j in range(i+1, n):
            c = A[j, i] / A[i, i];                 md_ops += 1
            A[j, i] = 0.0
            for k in range(i+1, n):
                A[j, k] -= c * A[i, k];            md_ops += 1; as_ops += 1
            b[j] -= c * b[i];                      md_ops += 1; as_ops += 1
    # backward substitution
    x = np.zeros(n)
    for i in range(n-1, -1, -1):
        s = b[i]
        for k in range(i+1, n):
            s -= A[i, k] * x[k];                   md_ops += 1; as_ops += 1
        x[i] = s / A[i, i];                        md_ops += 1
    return x, md_ops, as_ops

In [ ]:
print(f"{'n':>4} {'mult/div':>10} {'formula':>10} {'add/sub':>10} {'formula':>10} {'n^3/3':>10}")
for n in [2, 3, 4, 5, 10, 20, 40]:
    A = rng.standard_normal((n, n))
    b = rng.standard_normal(n)
    x, mdc, asc = gausselim_count(A, b)
    md_formula = n**3/3 + n**2 - n/3
    as_formula = n**3/3 + n**2/2 - 5*n/6
    print(f"{n:>4} {mdc:>10} {md_formula:>10.0f} {asc:>10} {as_formula:>10.0f} {n**3/3:>10.0f}")

The counts match the formulas exactly, and for large $n$ both are dominated by $n^3/3$. Compare with Cramer's rule from Part I:

In [ ]:
print(f"{'n':>4} {'Gaussian elimination ~ 2n^3/3':>30} {'Cramer ~ (n+1)!':>22}")
for n in [5, 10, 20, 30, 100]:
    print(f"{n:>4} {2*n**3/3:>30.3e} {float(math.factorial(n+1)):>22.3e}")

# Part VI. Several right-hand sides and the inverse

## 12. One elimination, many right-hand sides

All of our solvers accept `b` of shape `(n, m)`, so the wide augmented matrix $[A \mid b_1 \; b_2 \; \cdots \; b_m]$ costs one elimination plus $m$ backward substitutions.

**Example (slides: multiple right-hand sides).** Expected: $X_1 = (-1, 2, 0, 1)$ and $X_2 = (8/39, 19/39, -1/3, -3/13)$.

In [ ]:
A = np.array([[ 1.,  1.,  0.,  3.],
              [ 2.,  1., -1.,  1.],
              [ 3., -1., -1.,  2.],
              [-1.,  2.,  3., -1.]])
B = np.array([[ 4., 0.],
              [ 1., 1.],
              [-3., 0.],
              [ 4., 0.]])

X = gauss_pp(A, B)
print("X =\n", X)
print("\nexpected X2 =", np.array([8/39, 19/39, -1/3, -3/13]))

## 13. Computing $A^{-1}$ from $[A \mid I]$

The columns of $A^{-1}$ solve $AX = I$, so we pass the identity as the right-hand side.

**Example (slides).** Expected:
$$A^{-1} = \begin{bmatrix} 3/13 & 1/13 & 2/13\\ 7/13 & -2/13 & -4/13\\ 1/13 & 5/26 & -3/26\end{bmatrix}.$$

In [ ]:
def inverse(A):
    n = A.shape[0]
    return gauss_pp(A, np.eye(n))

A = np.array([[2.,  1.,  0.],
              [1., -1.,  4.],
              [3., -1., -2.]])
Ainv = inverse(A)
print("A^{-1} =\n", Ainv)
print("\n26 * A^{-1} =\n", 26 * Ainv)
print("\nA @ A^{-1} =\n", A @ Ainv)
print("\nmatches np.linalg.inv:", np.allclose(Ainv, np.linalg.inv(A)))

## 14. Don't use the inverse to solve $Ax = b$

Computing $A^{-1}$ costs about four times as much as one solve, and then $A^{-1}b$ is often *less* accurate. Try it on an ill-conditioned matrix (the Hilbert matrix):

In [ ]:
n = 12
H = 1.0 / (np.arange(1, n+1)[:, None] + np.arange(1, n+1)[None, :] - 1)
x_true = np.ones((n, 1))
bH = H @ x_true

x_solve = gauss_pp(H, bH)
x_inv = inverse(H) @ bH

print(f"cond(H)                       = {np.linalg.cond(H):.2e}")
print(f"solve directly : error = {rel_err(x_solve, x_true):.2e}, residual = {np.linalg.norm(H @ x_solve - bH):.2e}")
print(f"via inverse    : error = {rel_err(x_inv,   x_true):.2e}, residual = {np.linalg.norm(H @ x_inv   - bH):.2e}")

Both errors are large because $H$ is ill-conditioned, but notice that the direct solve has a residual near machine precision while the inverse-based one does not.

# Part VII. How does the cost grow with $n$? — timing experiments

The operation count predicts:

- substitution (forward/backward): $\;\sim n^2$ ⇒ doubling $n$ multiplies the time by about **4**;
- Gaussian elimination: $\;\sim n^3/3$ ⇒ doubling $n$ multiplies the time by about **8**.

We now test those predictions on your computer. We time four solvers:

| solver | what it is |
|---|---|
| `backward` | backward substitution (from Part II) |
| `gauss_pp` | our elimination with partial pivoting — row-by-row Python loop |
| `gauss_pp_vec` | the same algorithm, with each step done as one NumPy rank-1 update |
| `np.linalg.solve` | LAPACK's partial-pivoting LU, compiled and blocked |

All four do the same arithmetic (up to lower-order terms); they differ only in *how* it is executed.

In [ ]:
def gauss_pp_vec(A, b):
    """Partial pivoting, with the inner j-loop replaced by one outer-product update."""
    A = A.astype(float).copy()
    b = b.astype(float).copy()
    n = A.shape[0]
    for i in range(n-1):
        p = i + np.argmax(np.abs(A[i:, i]))
        if p != i:
            A[[i, p]] = A[[p, i]]
            b[[i, p]] = b[[p, i]]
        c = A[i+1:, i] / A[i, i]                     # all multipliers at once
        A[i+1:, i:] -= np.outer(c, A[i, i:])         # rank-1 update of the trailing block
        b[i+1:]     -= np.outer(c, b[i])
    return backward(A, b)


def time_it(f, *args, repeats=3):
    """Best wall-clock time of f(*args) over a few repeats, in seconds."""
    best = np.inf
    for _ in range(repeats):
        t0 = time.perf_counter()
        f(*args)
        best = min(best, time.perf_counter() - t0)
    return best

First, a sanity check that all four solvers agree:

In [ ]:
n = 200
A = rng.standard_normal((n, n))
b = rng.standard_normal((n, 1))
x_ref = np.linalg.solve(A, b)
for f in (gauss_pp, gauss_pp_vec):
    print(f"{f.__name__:>14}: relative difference from np.linalg.solve = {rel_err(f(A, b), x_ref):.2e}")

## 15. ✏️ Your turn: solve one large system

Change `n` below and re-run. Start at `n = 500`, then try `1000`, `2000`, … Before each run, **predict** the time from the previous one using the $n^3$ rule.

> Tip: if a cell takes too long, use *Kernel → Interrupt*.

In [ ]:
n = 1000          # <-- change me

A = rng.standard_normal((n, n))
b = rng.standard_normal((n, 1))
U = np.triu(A) + n * np.eye(n)      # a well-conditioned upper-triangular matrix

print(f"n = {n}")
print(f"  backward        : {time_it(backward, U, b):9.4f} s")
print(f"  gauss_pp        : {time_it(gauss_pp, A, b, repeats=1):9.4f} s")
print(f"  gauss_pp_vec    : {time_it(gauss_pp_vec, A, b):9.4f} s")
print(f"  np.linalg.solve : {time_it(np.linalg.solve, A, b):9.4f} s")
print(f"\n  predicted flops for elimination ~ 2n^3/3 = {2*n**3/3:.2e}")

If you prefer to type `n` when prompted, uncomment the first line of the next cell.

In [ ]:
# n = int(input("Enter n: "))
n = 1500

A = rng.standard_normal((n, n))
b = rng.standard_normal((n, 1))
t = time_it(gauss_pp_vec, A, b)
flops = 2 * n**3 / 3
print(f"n = {n}: gauss_pp_vec took {t:.3f} s  ->  about {flops / t / 1e9:.2f} GFLOPS")

## 16. A doubling experiment

Double $n$ repeatedly and look at the **ratio** of successive times. For an $O(n^p)$ algorithm the ratio tends to $2^p$.

In [ ]:
def doubling_table(solver, n_start, n_steps, make_input):
    print(f"solver: {solver.__name__}")
    print(f"{'n':>6} {'time (s)':>10} {'ratio':>7}")
    t_prev = None
    n = n_start
    for _ in range(n_steps):
        args = make_input(n)
        t = time_it(solver, *args, repeats=1 if n > 1000 else 3)
        ratio = "" if t_prev is None else f"{t / t_prev:7.2f}"
        print(f"{n:>6} {t:>10.4f} {ratio:>7}")
        t_prev = t
        n *= 2

def square_system(n):
    return rng.standard_normal((n, n)), rng.standard_normal((n, 1))

def triangular_system(n):
    return np.triu(rng.standard_normal((n, n))) + n * np.eye(n), rng.standard_normal((n, 1))

doubling_table(gauss_pp_vec, 125, 5, square_system)
print()
doubling_table(backward, 250, 5, triangular_system)

**Questions.**
1. Which ratio do you see for `gauss_pp_vec`? For `backward`? Do they approach $8$ and $4$?
2. For small $n$ the ratios are often *smaller* than predicted. Why? (Think about the fixed overhead of each Python loop iteration versus the arithmetic it does.)
3. `backward` runs a Python loop of only $n$ iterations, each doing $O(n)$ arithmetic in NumPy. At these sizes the per-iteration overhead (a few microseconds) outweighs the arithmetic, so its time grows almost like $n$ and the ratio sits near $2$, not $4$. How large must $n$ be before the $n^2/2$ arithmetic takes over? Try it — but watch the memory: an $n \times n$ matrix of doubles takes $8n^2$ bytes.

## 17. Log–log plot: reading off the exponent

If $T(n) \approx C n^p$, then $\log T = \log C + p \log n$ — a straight line of slope $p$ on a log–log plot. We fit $p$ by least squares on the larger values of $n$.

✏️ **Your turn:** edit `ns` to push to larger $n$. The pure-loop `gauss_pp` is slow, so it gets its own, shorter list.

In [ ]:
ns      = [100, 200, 400, 800, 1200, 1600]     # <-- change me
ns_loop = [100, 200, 400, 800]                 # gauss_pp is slow; keep this list short

results = {"backward": [], "gauss_pp": [], "gauss_pp_vec": [], "np.linalg.solve": []}
for n in ns:
    A, b = square_system(n)
    U, _ = triangular_system(n)
    results["backward"].append(time_it(backward, U, b))
    results["gauss_pp_vec"].append(time_it(gauss_pp_vec, A, b))
    results["np.linalg.solve"].append(time_it(np.linalg.solve, A, b))
    if n in ns_loop:
        results["gauss_pp"].append(time_it(gauss_pp, A, b, repeats=1))
    print(f"n = {n} done")

In [ ]:
def fit_exponent(ns, ts, last=3):
    # slope of log t versus log n over the last few points
    p, _ = np.polyfit(np.log(ns[-last:]), np.log(ts[-last:]), 1)
    return p

fig, ax = plt.subplots(figsize=(7, 5))
markers = {"backward": "o", "gauss_pp": "s", "gauss_pp_vec": "^", "np.linalg.solve": "D"}
for name, ts in results.items():
    x_n = ns_loop if name == "gauss_pp" else ns
    p = fit_exponent(x_n, ts)
    ax.loglog(x_n, ts, marker=markers[name], label=f"{name}  (slope ≈ {p:.2f})")

# reference slopes, anchored at the gauss_pp_vec curve
n_ref = np.array([ns[0], ns[-1]], dtype=float)
t0 = results["gauss_pp_vec"][0]
ax.loglog(n_ref, t0 * (n_ref / n_ref[0])**3, "k--", lw=1, label="reference slope 3")
ax.loglog(n_ref, t0 * (n_ref / n_ref[0])**2, "k:",  lw=1, label="reference slope 2")

ax.set_xlabel("n")
ax.set_ylabel("time (s)")
ax.set_title("Cost of direct solvers versus n")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.show()

**Interpreting the plot.**

- The lines for the three elimination solvers have similar slopes for large $n$: same algorithm, same $O(n^3)$ growth. The vertical gaps between them are the *constant* $C$ — interpreted Python loops versus vectorized NumPy versus compiled, cache-blocked LAPACK.
- `backward` grows much more slowly — its operation count is $O(n^2)$, and at these sizes its measured slope is even lower because loop overhead dominates. Either way a triangular solve is cheap next to an elimination, which is why we are happy to do many triangular solves once a matrix is factored.
- Measured slopes for `np.linalg.solve` are often below 3 at moderate $n$: LAPACK is so fast that overheads and memory traffic still matter. Push `ns` to a few thousand and the slope climbs toward 3.

## 18. ✏️ Your turn: predict, then measure

Use the fitted model $T(n) \approx C n^p$ to predict how long `np.linalg.solve` would take for a size you choose, then (if it is reasonable) measure it.

In [ ]:
name = "np.linalg.solve"
p, logC = np.polyfit(np.log(ns[-3:]), np.log(results[name][-3:]), 1)
C = np.exp(logC)

n_big = 4000        # <-- change me

print(f"model: T(n) ≈ {C:.3e} * n^{p:.2f}")
print(f"predicted time for n = {n_big}: {C * n_big**p:.3f} s")
print(f"predicted time for n = {10*n_big}: {C * (10*n_big)**p:.1f} s   (and memory for A: {8*(10*n_big)**2/1e9:.1f} GB)")

In [ ]:
# Only run this if the prediction above is a few seconds or less!
A, b = square_system(n_big)
t = time_it(np.linalg.solve, A, b, repeats=1)
print(f"measured time for n = {n_big}: {t:.3f} s")

# Exercises

1. **Cost of the inverse.** Time `inverse(A)` (i.e. `gauss_pp(A, np.eye(n))`) against `gauss_pp(A, b)` for a single `b` for several $n$. Is the ratio close to the slides' prediction of about 4? (Use `gauss_pp_vec` with `np.eye(n)` for larger $n$.)
2. **Many right-hand sides.** For fixed $n = 1000$, time `gauss_pp_vec(A, B)` with `B` of shape `(n, m)` for $m = 1, 10, 100, 1000$. Explain the results using the count "one elimination plus $m$ backward substitutions."
3. **Growth of the pivots.** Modify `gausselim` to record $\max_{i,j} |a_{ij}|$ after every elimination step. Compare the growth for naive elimination and partial pivoting on `A_bad` from Section 10.
4. **Scaled pivoting cost.** Verify experimentally that `gauss_spp` costs essentially the same as `gauss_pp` for large $n$. Why is computing the scales once only $O(n^2)$ extra work?
5. **Keep the multipliers.** Modify `gauss_pp_vec` to store each multiplier `c` in the zeroed-out part of `A`, and return `L`, `U`, and `IP`. Check that `A[IP] ≈ L @ U`. This is the LU factorization — the subject of the next lecture.